(chpt-arcpy)=
# Die Handelsbibliothek

Geospatialanalysen mit ESRIs kommerzieller ArcGIS-Software und Arcpy-Paket. Dieses Notebook kann nicht in Jupyter ausgeführt werden und Codeblöcke erfordern die kommerzielle Arcpy-Bibliothek von ESRI.


## Einleitung

Die *ArcGIS Pro* Software von *ESRI* verfügt über eine eigene Conda-Umgebung, die die Zusammenarbeit mit der kommerziellen `arcpy`Bibliothek ermöglicht. `arcpy` ist entweder direkt unter `ArcGIS Pro` oder unter Verwendung von ESRIs conda-Umgebung zugänglich, die über ArcGIS Pro verwaltet werden kann. Da `arcpy` an die Nutzung eines kommerziellen Umfelds gebunden ist, ist es nur in jupyter Notebooks, die direkt unter [esri.com](https://notebooks.esri.com/) (diese werden aber nicht mit den Beispielen in diesem Abschnitt funktionieren) zu erreichen. Diese Seite erklärt die grundsätzliche Nutzung von `arcpy` in extern {ref}`IDEs <sec-ide>` (z.B. PyCharm, VS Code oder Spyder), die Integration von Lizenzen und die grundlegende Funktionalität von `arcpy`.

```{admonition} Requirements
Achten Sie darauf, den Unterschied zwischen Rastern und Formdateien zu verstehen, wie im Abschnitt {ref}`geospatial data <geospatial-data>` erläutert.
```

```{note}
Die Entwicklung von **ArcMap** mit ihrem *Python2* ausführbar ** wird unterbrochen**. Aus diesem Grund wird hier nur die Nutzung von *ArcGIS Pro* mit *Python3*-Umgebung erläutert.
```

```{admonition} Linux Users
:class: warning
GESELLSCHAFT Pro und die Wrapper, die die `arcpy` Bibliothek sind ** Nur Windows*-Anwendungen. Zum Zeitpunkt des Schreibens dieser Einführung können *ArcGIS Pro* und `arcpy` nicht auf *Linux* oder *macOS* Plattformen verwendet werden.
```

## Hintergrund

***Warum kann mit `arcpy` in Python arbeiten, obwohl es viele Lizenz- und Plattformbeschränkungen gibt?***

Wenn Sie für ein Unternehmen mit Engineering-Services arbeiten, nutzt das Unternehmen wahrscheinlich ArcGIS wegen seiner Popularität und kommerziellen Support-Service. In diesem Fall umfasst ein Unternehmen (oder Forschungseinrichtung) die Lizenzgebühren und es ist ebenso wahrscheinlich, dass ein Windows-Betriebssystem aus ähnlichen Gründen verwendet wird. Unter den beliebten Funktionalitäten der ArcGIS-Software sind *Raster Calculator*, *Shapefile Feature* Manager oder Werkzeuge für die statistische Analyse von Geo-Datenbanken. Dank `arcpy` können solche beliebten ArcGIS-Tools in Python-Skripte eingebettet werden, wodurch Workflows automatisiert und Effizienz deutlich gesteigert werden können.

Diese Seite zeigt die Grundlagen für die Verarbeitung von Raster- und Formdateidaten in Python mit `arcpy`. Es gibt viele weitere Methoden, die in `arcpy` umgesetzt werden, und hier werden nur die Grundlagen erläutert.

## Verwenden Sie `arcpy` mit externen IDEs

### Setup Interpreter

Der Python-Installationsabschnitt erklärt {ref}`how to set up PyCharm IDE <ide-setup>` mit einer conda-Umgebung. Um eine neue conda-Umgebung in PyCharm mit der conda-Umgebung von ESRI zu schaffen, muss die **Location** anders definiert werden (siehe die [original screenshot](https://raw.githubusercontent.com/sschwindt/hydroinformatics/main/docs/img/pyc-prj-setup.png)):

* Wenn *ArcGIS * Pro wurde weltweit vom Systemadministrator installiert, verwenden: `%PROGRAMFILES%\ArcGIS\Pro\bin\Python\Scripts\propy`
* Wenn *ArcGIS * Pro wurde für einzelne Benutzer installiert, verwenden: `%LOCALAPPDATA%\Programs\ArcGIS\Pro\bin\Python\Scripts\propy`

Es kann vorkommen, dass es nicht notwendig ist, das `propy`-Verzeichnis hinzuzufügen. Darüber hinaus finden Sie die `conda.exe` oder `python.exe` in den obigen Verzeichnissen und definieren sie im Feld **Conda ausführbar**. Vervollständigen Sie die Schaffung der neuen Umgebung mit einem Klick auf *Kreate*.

Um weitere Pakete zu installieren, folgen Sie den [Beschreibungen von Esri](https://developers.arcgis.com/python/guide/install-and-set-up/#Step-1:-Get-Conda).

```{tip}
Wenn Sie mit den obigen Anweisungen kämpfen, werfen Sie einen Blick auf die [Entwickler's docs on run stand-alone scripts](https://pro.arcgis.com/en/pro-app/arcpy/get-started/using-conda-with-arcgis-pro.htm) (siehe Anleitungen unter `propy` anstatt `python27`).
```

### Import Arcpy und seine Module
`arcpy` kommt mit eigenen Klassen für geospatiale Datenobjekte (z.B. `arcpy.Raster` für netzgebundene Daten) und Module für Mapping (`arcpy.mp`), Emulation des *Spatial Analyst* (`arcpy.sa`) oder Zugriff auf Daten (`arcpy.da`). Eine vollständige Liste finden Sie auf der [developer's website](https://pro.arcgis.com/en/pro-app/arcpy/get-started/importing-arcpy.htm). Diese Seite enthält Code-Blöcke mit `arcpy` und `arcpy.sa`, bei denen *Spatial Analyst*-Objekte mit `*` importiert werden, um einen direkten Zugriff auf z.B. `Con()` (anstelle von `arcpy.sa.Con()`) zu ermöglichen, was der bedingten Aussage von `arcpy` für Raster entspricht.

In [ ]:
import arcpy
from arcpy.sa import *

### Setup arcpy Workspace

Geospatiale Berechnungen können viele Nebenprodukte produzieren, die schwer sein können. Um die Daten besser zu kontrollieren, die von `arcpy` generiert werden, und wo es eine gute Idee ist, einen Workspace für jedes Arcpy-Skript zu definieren:

In [ ]:
arcpy.env.workspace = "C:\\workspace\\"  # or use os.path.dirname(__file__) to go to the script directory

```{important}
Vermeiden Sie Leerzeichen im Workspace-Verzeichnis.
```

Es kann auch nützlich sein, das Überschreiben bereits bestehender Dateien mit dem gleichen Namen aufgrund der Dateigröße zu aktivieren. Dieses Verhalten kann oder darf nicht erwünscht sein und kann wie folgt gesteuert werden.

In [ ]:
arcpy.gp.overwriteOutput = True   # enable overwriting
arcpy.gp.overwriteOutput = False  # disable overwriting

### Set Spatial Extents

Geospatial-Datensätze können große Ausmaße haben, ohne dass Daten in großen Teilen geschrieben wurden. Die Verarbeitung von Datenzellen kann zu unnötig langer Rechenzeit führen. Daher ist es ratsam, den Berechnungsumfang mit:

In [ ]:
arcpy.env.extent = "MAXOF"  # uses the combined extent of all input datasets
arcpy.env.extent = "MINOF"  # uses only the overlap of all input datasets
arcpy.env.extent = arcpy.Extent(arcpy.Raster("base.tif"))  # uses the controlled extent of a raster
arcpy.env.extent = "Xmin YMin XMax Ymax"  # imposes user-defined minimum and maximum coordinates

(licenses)=
### Checkout-Lizenzen

Viele `arcpy` Methoden benötigen Lizenzen wie *Spatial Analyst* oder *3D*. In eigenständigen Skripten können Lizenzen (*checked out*) mit `arcpy.CheckOutExtension('NAME')` und deaktiviert (*checked in*) mit `arcpy.CheckInExtension('NAME')` aktiviert werden. Angesichts der Objektorientierung sollten `arcpy`-Operationen in Funktionen oder Klassenmethoden eingebettet werden. Daher wird empfohlen, Funktionen oder Methoden mit `arcpy` mit {ref}`decorators <wrappers>` zu wickeln, die notwendige Lizenzen aktivieren. Der folgende Codeblock bietet eine Wrapper, um eine *Spatial Analyst* Lizenz für eine Funktion zu aktivieren.

In [ ]:
def spatial_license(func):
    def wrapper(*args, **kwargs):
        arcpy.CheckOutExtension('Spatial')
        result = func(*args, **kwargs)
        arcpy.CheckInExtension('Spatial')
        return result
    return wrapper

(arcpy-errors)=
### Fehler verfolgen

Der Abschnitt {ref}`Python Errors, Logging, and Debugging <sec-pyerror>` bietet nützliche Anweisungen für Fehlerbehebungen in Code- oder Codenutzung. Um Probleme in objektorientierten `arcpy`Scripts zu identifizieren, wird eine zusätzliche Wrapper-Funktion empfohlen, die `arcpy`Fehler an eine Logdatei (`logger`) schreibt. Die `logger.*(MESSAGE)`-Ausdrücke können auch durch `print(MESSAGE)` ersetzt werden.

In [ ]:
import logging


def err_info(func):
    def wrapper(*args, **kwargs):
        arcpy.gp.overwriteOutput = True
        logger = logging.getLogger("logfile")
        try:
            return func(*args, **kwargs)
        except arcpy.ExecuteError:
            logger.info(arcpy.GetMessages(2))
            arcpy.AddError(arcpy.GetMessages(2))
        except Exception as e:
            logger.info(e.args[0])
            arcpy.AddError(e.args[0])
        except:
            logger.info(arcpy.GetMessages())
    return wrapper

## Raumanalyst und Rasteroperationen

### Grundlagen

`arcpy` bietet verschiedene Optionen, um Geospatial zu laden [rasters](https://pro.arcgis.com/en/pro-app/arcpy/classes/raster-object.htm) (gridded data), die verschiedene Formate wie *Esri Grid* haben kann (kein Ende ist der Raster ein Ordner mit anderen Dateien), {term}`GeoTIFF`, *DAT* und vieles mehr. Das folgende Skript lädt eine Fließtiefe Grid raster`h` und eine Strömungsgeschwindigkeit Grid raster `u`:

In [5]:
h = arcpy.Raster("geodata/input/rasters/h")
u = arcpy.Raster("geodata/input/rasters/u")

Der {term}`Froude number` kann aus der Strömungstiefe und Geschwindigkeit und der Gravitationskonstanten $g$=9.81 m/s$^2$ pixel berechnet werden. Das folgende Skript berechnet die Froude-Nummer für alle Pixel, bei denen die Durchflusstiefe mindestens 0,1 m beträgt. Der Rastervergleich wird durch die *Spatial Analyst*'s `Con(if_condition, then, else)`Zustandserklärung erreicht. Die beiden Raster (`u` und `h`) werden als `arcpy.sa.Float()`Objekte weitergegeben, um sicherzustellen, dass das Skript den richtigen Pixeldatentyp verwendet.

In [ ]:
froude = Con(Float(h) > 0.1, Float(u) / SquareRoot(Float(h) * Float(9.81)))

Besser als das 0,1-Meter-Kriterium ist die Berechnung der Froude-Zahl überall dort, wo die Strömungstiefe und die Geschwindigkeit einen Zahlenwert haben. `arcpy.sa.IsNull()` wertet aus, wo Pixel nicht numerisch sind. Wir interessieren uns jedoch für das Gegenteil (d.h. Pixel, die nicht numerisch sind), das wir durch das `~` (nicht)-Zeichen erhalten. Die unterfunktionalisierte Berechnung der Froude-Nummer nutzt die Möglichkeit, mehrere `Con()`-Ausdrücke zu verschachteln, um beide Raster (`u` und `h`) für numerische Pixel zu überprüfen.
Darüber hinaus benötigen wir eine Spatial Analyst Lizenz, um dieses Skript auszuführen. Daher ist es sinnvoll, den obigen Codeblock in eine Funktion umzuschreiben, die die `spatial_license`Wrapper-Funktion aus dem oben genannten {ref}`checkout Licenses <licenses>`-Bereich verwendet. Um den Code robust zu machen, fügen wir auch die {ref}`above-defined *err_info* <arcpy-errors>`Wrapper-Funktion hinzu.

In [ ]:
@err_info
@spatial_license
def calculate_froude(h, u):
    return Con(~IsNull(h), Con(~IsNull(u), Float(u) / SquareRoot(Float(h) * Float(9.81))))

froude = calculate_froude(h, u)

Erfahren Sie mehr über *Raster Calculator* und *Map Algebra* auf der [developer's website (esri)](https://pro.arcgis.com/en/pro-app/tool-reference/image-analyst/raster-calculator.htm).

```{tip}
Die Funktionen mit *Raster Calculator* können auch mit den Open-Access-Bibliotheken `gdal` und `numpy` ausgeführt werden. Alles, was Sie tun müssen, ist:

1. Lesen Sie {ref}`raster as array <createarray>`.
2. Verwenden Sie {ref}`numpy <array-matrix-operations>` und/oder {ref}`pandas <pandas>`, um typische *Raster Calculator*-Operationen und viele (viele) mehr auf dem Array auszuführen.
3. Schreiben Sie die {ref}`array back to a raster <create-raster>`.
```

### Zellstatistik

Bei der Auswertung von numerischen Modelldaten werden oft statistische Werte (z.B. minimal oder maximal) eines oder mehrerer Raster berechnet. Der Vergleich mehrerer ähnlicher Raster ist z.B. dann sinnvoll, wenn derselbe Parameter mit zwei verschiedenen Modellen oder zu unterschiedlichen Zeitpunkten berechnet wurde (z.B. zur Beurteilung der morphodynamischen Flussentwicklung). `arcpy.sa.CellStatistics([Raster1, Raster2, ... RasterN], TYPE, ...)` ermöglicht solche statistischen Auswertungen. Der folgende Codeblock verdeutlicht den Vergleich der mit zwei verschiedenen hydrodynamischen numerischen Modellen berechneten Strömungsgeschwindigkeiten durch die Berechnung der `MEAN` (Durchschnitt) und Standardabweichung (`STD`).

In [ ]:
u_basement = arcpy.Raster("geodata/bm/velocity.tif")
u_tuflow = arcpy.Raster("geodata/tf/velocity.dat")

u_mean = CellStatistics([u_basement, u_tuflow], "MEAN")
u_stdv = CellStatistics([u_basement, u_tuflow], "STD")

Lesen Sie mehr Optionen Statistiktypen und Umgang mit nicht-numerischen Daten auf der [developer's website (Esri)](https://pro.arcgis.com/en/pro-app/tool-reference/spatial-analyst/cell-statistics.htm).

```{tip}
Eine Open Access Alternative zu `arcpy`'s `CellStatistics` ist die {ref}`rasterstats library <zonal>` (Verwendung: `rasterstats.zonal_stats(zone, raster_file_name, stats=['min', 'max', 'median', 'majority', 'sum', '...many more...'])`).
```

## Shapefile Operationen

Das geospatial {ref}`shapefile <chpt-shp>` Vektorformat ist eine Esri-Erfindung. Kein Wunder, `arcpy` ist gut bei der Verarbeitung dieses Vektordatenformats. In der Hydrauliktechnik erstellen wir jedoch in der Regel (Zeichnen) Shapefiles manuell entweder direkt mit ArcGIS oder dessen Open-Source-Konkurrenten [QGIS](https://www.qgis.org/) um z.B. bestimmte Strömungsregionen abzugrenzen. Beispiele sind {ref}`in the BASEMENT tutorial <chpt-basement>` (erweitern Sie die Erstellung von Elevationspunkt, Randpolygon und Breakline Polyline Shapefiles). In Codes wird die Verarbeitung von Shapefiles nur bei der Analyse der Ausgabe von numerischen Modellen wichtig (z.B. zur Klassifizierung von morphologischen Einheitsmerkmalen, zur exakten Berechnung von Patch-Bereichen oder zur automatischen Versteifung in Bauplänen). In diesem Stadium müssen Rasterdaten (Ausgang von numerischen Modellen) zunächst in Formdateien umgewandelt werden. Aus diesem Grund beginnt dieses Tutorial mit der Umwandlung von Rasterdaten in Formdateien zusammen mit der Darstellung anderer Funktionen wie der Berechnung von Patch-Bereich und dem Zugriff auf formfile Attributtabellen.

Raster können in Polygon und andere Shapefile-Typen (z.B. Punkt) umgewandelt werden. Das folgende Beispiel zeigt die Umwandlung eines Rasters in eine Polygonformdatei. Es verwendet einen ganzzahligen Raster aller Pixel, wobei die Strömungstiefe und -geschwindigkeit kleiner als 1,4 m bzw. 0,15 m/s sind. Diese flachen und langsam fließenden Regionen werden als *slackwater* (nach {cite:t}`wyrick_geospatial_2014`) bezeichnet. Da Slackwater als bevorzugter Lebensraum einiger Wasserarten bezeichnet wird, fragen wir uns jetzt, wie viel Slackwater-Bereich das numerische Modell im simulierten Flussabschnitt vorhersagt. Dazu wandeln wir den Slackwater-Raster in eine Formdatei um und berechnen die Oberfläche der Formdatei mit den folgenden `arcpy` Methoden:

* Convert the raster to a shapefile with [`arcpy.RasterToPolygon_conversion()`](https://pro.arcgis.com/en/pro-app/tool-reference/conversion/raster-to-polygon.htm) with arguments:
    - `in_raster` is an `arcpy.Raster()` object of **integer** values (using a `Float` raster results in an error!).
    - `out_polygon_features` is a *string* of the output file name and directory.
    - `simplify` is an optional *string* that can be either `"NO_SIMPLIFY"` to force exact drawing of polygon boundaries along pixel border, or `"SIMPLIFY"` to enable polygon boundaries crossing pixels.
* Add a new field to the new polygon shapefile with [`arcpy.AddField_management()`](https://pro.arcgis.com/en/pro-app/tool-reference/data-management/add-field.htm) with arguments:
    - `field_name` can be any *string* without blanks.
    - `field_type` is a *string* defining if the field is numeric (e.g., `"FLOAT"` or `"LONG"` for *integer*), date/time (`"DATE"`), `"TEXT"`, or `"RASTER"`.
    - `field_precision` is an (optional) *integer* (*Long*) defining the number of digits that can be stored in the new field.
    - More optional arguments can be set to define the number of decimals or characters, an alternative field name, enable `NULL`, a field domain, or if a field is required.
* Calculate patch area with [`arcpy.CalculateGeometryAttributes_management()`](https://pro.arcgis.com/en/pro-app/tool-reference/data-management/calculate-geometry-attributes.htm) with arguments:
    - `in_features` is a *string* defining the directory and name of a feature layer.
    - `geometry_property` is a nested list of `[[Target-field-name, Property], [Another-target-field-name, Another-property], ...]` for calculating geometric properties such as `"AREA"`, `"HOLE_COUNT"`, or `"PART_COUNT"` (and many more).
    - `area_unit` can be `"SQUARE_METERS"` or `"SQUARE_KILOMETERS"` (and many other options for U.S. customary units).
    - More optional arguments can be set to define length units (for perimeter assessments), or a coordinate system and format.
    
Der untenstehende Codeblock enthält zusätzlich die Anwendung dieser Methoden und zeigt auch, wie der Bereichswert aus der Attributtabelle einer Shapefile mit `arcpy.da.UpdateCursor(shapefile-name, field-name)` reihenweise gelesen werden kann.

In [ ]:
# create a slackwater raster that is arcpy.sa.Int(1) where h and u criteria are true and NULL elsewhere
slackwater = Con((Float(h) <= 1.4) & (Float(u) <= 0.15), Int(1))


# define directory and name of the new shapefile
new_shp_file = "geodata/shapefiles/slackwater.shp"
# convert slackwater raster to polygon shapefile
arcpy.RasterToPolygon_conversion(in_raster=slackwater, out_polygon_features=new_shp_file, simplify="NO_SIMPLIFY")
# add a new field to the new shapefile's attribute table (name)
arcpy.AddField_management(new_shp_file, field_name="F_AREA", field_type="FLOAT", field_precision=9)
# calculate area of all polygons in attribute table
area_unit="SQUARE_METERS"
arcpy.CalculateGeometryAttributes_management(in_features=new_shp_file, geometry_property=[["F_AREA", "AREA"]], area_unit=area_unit)

area = 0.0
with arcpy.da.UpdateCursor(new_shp_file, "F_AREA") as cursor:
    for row in cursor:
        try:
            area += float(row[0]) 
        except ValueError:
            print("WARNING: Patch with invalid area value (%s)." % str(row))

print("Sum of all patches = {0} {1}".format(str(area), area_unit))

```{warning}
Berechnen Sie niemals direkt Bereiche aus Rastern, indem Sie die Anzahl (Menge) von Pixeln multiplizieren, die ein bestimmtes Kriterium mit der Pixelgröße erfüllen (z.B. 1mx1m). Diese Berechnung scheitert in der Praxis oft an fehlerhaften internen Zuordnungen von Zellgrößen, die kaum robust gesteuert werden können (insbesondere beim Schalten zwischen US-üblichen und S.I.-Einheiten).
```

```{tip}
Shapefile Handling, Ableitung geometrischer Eigenschaften und viele weitere Operationen können auch mit der `ogr` Bibliothek durchgeführt werden, die zusammen mit `gdal` kommt. Lesen Sie mehr im Abschnitt {ref}`shapefile handling <chpt-shp>`.
```